In [1]:
import json
from pathlib import Path
from typing import List, Dict, Any
import sys
sys.path.append("../../")

from src.config import RAW_DATA_DIR, METADATA_INDEX, CONTENT_INDEX, PROJECT_ROOT, setup_logger
from src.services import OpenSearchClient
from src.ingestion.processors import (
    process_json_dolphin,
    extract_article_metadata,
)
from src.ingestion.processors.embedding_processor import VectorProcessor
from src.ingestion.processors.chunk_preprocessor import ContentAggregator


In [2]:
def load_dolphin_json(json_path: Path) -> Dict[str, Any]:
    """Carrega um arquivo JSON do Dolphin."""
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_opensearch_mapping(mapping_path: Path) -> Dict[str, Any]:
    """Carrega o arquivo de mapping do OpenSearch."""
    with open(mapping_path, 'r', encoding='utf-8') as f:
        return json.load(f)

#### process_all_jsons(json_dir: Path) -> tuple[List[Dict], List[Dict]]:

<hr>

In [10]:
# def process_all_jsons(json_dir: Path) -> tuple[List[Dict], List[Dict]]:
"""
Processa todos os JSONs do Dolphin em um diretório.

Returns:
    Tupla (metadados, chunks_conteudo)
"""
# input
json_dir = RAW_DATA_DIR / "json_extraction"
all_metadata = []
all_chunks = []

json_files = list(json_dir.glob("*.json"))
print(f"Encontrados {len(json_files)} arquivos JSON")

Encontrados 2 arquivos JSON


In [11]:
dolphin_json = load_dolphin_json(json_files[0])

# Gera chunks
metadata = extract_article_metadata(dolphin_json, dolphin_json["source_file"])

2026-03-10 18:54:56 - src.ingestion.processors.metadata_extractor - INFO - book_id='atoms_confusion.pdf': Abstract concatenated from 3 parts
2026-03-10 18:54:56 - src.ingestion.processors.metadata_extractor - INFO - book_id='atoms_confusion.pdf': Metadata extracted successfully


In [12]:
metadata

{'book_id': '_ICSA_2026__Argus_Architecture.pdf',
 'title': 'ARGUS: A Context-Aware Software Architecture\nfor Smart Environments',
 'abstract': 'Abstract — The paradigm of Smart Environment (SE) has tran-\nsitioned from simple remote automation to proactive, context-\naware ecosystems driven by Artificial Intelligence. However,\nSecuring Smart Environments requires an intelligence-based and\nhigh-frequency sensor streams to computationally intensive Large\nLanguage Models (LLMs), imposes significant challenges regard-\ning latency, interoperability, and data privacy. To address these\nissues, we present ARGUS, a distributed, event-driven software\npackage implemented as part of the larger open source community\nacross the Edge-Cloud continuum. We validate the architecture\nthrough a reference implementation and performance evaluation,\ndemonstrating that ARGUS effectively accommodates heavy\ncomputational workloads, such as those used in generative AI\nenvironments, by providing smart

In [12]:
all_metadata = []
all_chunks = []

for json_file in json_files:
    dolphin_json = load_dolphin_json(json_file)

    # file name
    book_id = dolphin_json["source_file"]
    print(f"Processando: {book_id}")

    # Gera chunks
    chunks = process_json_dolphin(dolphin_json, book_id)
    all_chunks.extend(chunks)
    
    # Extrai metadados
    metadata = extract_article_metadata(dolphin_json, book_id)
    if metadata:
        all_metadata.append(metadata)
    
    

print(f"Total: {len(all_metadata)} metadados, {len(all_chunks)} chunks")
    # return all_metadata, all_chunks

Processando: atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ingestion.processors.chunk_builder - WARNING - Unknown tag: type='reference' at book_id=atoms_confusion.pdf
2026-03-10 18:55:08 - src.ing

In [11]:
all_chunks[160]

{'book_id': '_ICSA_2026__Argus_Architecture.pdf',
 'sec_0': 'ARGUS: A Context-Aware Software Architecture\nfor Smart Environments',
 'sec_1': 'V. ARGUS subsystems',
 'sec_2': 'A. Real-time Recommendation Eng',
 'section_context': 'V. ARGUS subsystems > A. Real-time Recommendation Eng',
 'doc_type': 'text',
 'page_number': 5,
 'reading_order': 10,
 'content': 'V. ARGUS subsystems > A. Real-time Recommendation Eng\n\nUpon consumption, the engine transforms the continuous\nevent stream into a structured format suitable for supervised\nlearning. Incoming events are converted into tabular records,\nwhere devices are categorized as either predictors (sensors)\nor control targets (actuators). These records are enriched via\ntemporal transformation , which encodes timestamps into fea-\ntures ( e.g., time-of-day) and lagged states to capture sequential\nusage patterns. To handle the irregular frequency of IoT data,\nthe system employs dynamic window construction . Instead of\nfixed time interva

In [2]:
aggregator = ContentAggregator()

2026-03-10 18:46:20 - src.services.llm_factory - INFO - Criando cliente OpenAI com modelo: gpt-4o


In [3]:
fig_str = "![Figure](figures/atoms_confusion_page_001_figure_000.png)"

In [6]:
filtered_path = aggregator._find_url(fig_str)
print(filtered_path)

figures/atoms_confusion_page_001_figure_000.png


In [7]:
aggregator._resolve_image_path(filtered_path)

'C:\\Users\\Erlonidas\\Documents\\deploys\\rag_book_challenge\\data\\raw\\figures\\atoms_confusion_page_001_figure_000.png'

In [13]:
enriched_chunks = aggregator.centralize_section_context_for_element(all_chunks)

In [19]:
all_figs = []
all_tabs = []
for chunk in enriched_chunks:
    if chunk["doc_type"] == "figure":
        all_figs.append(chunk)
    elif chunk["doc_type"] == "table":
        all_tabs.append(chunk)
print(str(len(all_figs)) + " figures")
print(str(len(all_tabs)) + " tables")

18 figures
3 tables


In [31]:
all_figs[2]

{'book_id': 'atoms_confusion.pdf',
 'sec_0': 'Atoms of Confusion: The Eyes Do Not Lie',
 'sec_1': '2   EXAMPLE',
 'sec_2': '',
 'section_context': '2   EXAMPLE',
 'doc_type': 'figure',
 'page_number': 4,
 'reading_order': 3,
 'content': '2   EXAMPLE\n\n<VISUAL_DESCRIPTION>**Figure 1: Transformation of a C function `print_int` to remove an atom of confusion, specifically the Conditional Operator. The left panel shows the original code with nested ternary operators, which exemplifies the atom of confusion. The right panel displays the refactored version, where the nested ternary operators are replaced with a more straightforward `if-else` structure, enhancing code clarity and reducing potential misunderstandings. This transformation aims to improve code comprehension by eliminating complex conditional expressions.**\n</VISUAL_DESCRIPTION>',
 'image_metadata': 'C:\\Users\\Erlonidas\\Documents\\deploys\\rag_book_challenge\\data\\raw\\figures\\atoms_confusion_page_004_figure_002.png'}

In [32]:
print(all_figs[2]["content"])

2   EXAMPLE

<VISUAL_DESCRIPTION>**Figure 1: Transformation of a C function `print_int` to remove an atom of confusion, specifically the Conditional Operator. The left panel shows the original code with nested ternary operators, which exemplifies the atom of confusion. The right panel displays the refactored version, where the nested ternary operators are replaced with a more straightforward `if-else` structure, enhancing code clarity and reducing potential misunderstandings. This transformation aims to improve code comprehension by eliminating complex conditional expressions.**
</VISUAL_DESCRIPTION>


In [35]:
print(all_figs[2]["image_metadata"])

C:\Users\Erlonidas\Documents\deploys\rag_book_challenge\data\raw\figures\atoms_confusion_page_004_figure_002.png


In [36]:
vector_processor = VectorProcessor()

2026-03-10 19:27:50 - src.ingestion.processors.embedding_processor - INFO - Embed Model successfully loades. Its dimensions is: 3072


In [37]:
chunks_with_embeddings = vector_processor.add_vectors_to_chunks(enriched_chunks)

2026-03-10 19:44:02 - src.ingestion.processors.embedding_processor - INFO - Generating embeddings for 217 docs...


In [39]:
chunks_with_embeddings[100]

{'book_id': 'atoms_confusion.pdf',
 'sec_0': 'Atoms of Confusion: The Eyes Do Not Lie',
 'sec_1': '7 RELATED WORK',
 'sec_2': '',
 'section_context': '7 RELATED WORK',
 'doc_type': 'text',
 'page_number': 10,
 'reading_order': 14,
 'content': "7 RELATED WORK\n\nGopstein et al. [4] have performed an experiment with 73 par-\nticipants and showed empirically that these atoms of confusion\ncould lead to a significantly increased rate of misunderstanding\nwhen compared with code without atoms of confusion. As they\nwere expected, the results were consistent: 70 out of 100, 92.5 %\nabout the behavior of a piece of code. These different conclusions\ncan naturally lead to bugs. Still, according to Gopstein et al. [4] ,\natoms of confusion can also cause diminished productivity, faulty\nproducts, and higher costs. They have also cataloged 15 atoms of\nconfusion and provided a methodology for empirically deriving\nthese complex atoms. In their study [4] , they have conducted two\nexperiments. On

In [42]:
mapping_path = PROJECT_ROOT / "src" / "config" / "opensearch_chunk_search_schema.json"
content_mapping = load_opensearch_mapping(mapping_path)
content_mapping["mappings"]["properties"]["vetor"]["dimension"] = vector_processor.dimension


In [43]:
content_mapping

{'mappings': {'properties': {'book_id': {'type': 'keyword'},
   'sec_0': {'type': 'keyword'},
   'sec_1': {'type': 'keyword'},
   'sec_2': {'type': 'keyword'},
   'section_context': {'type': 'keyword'},
   'doc_type': {'type': 'keyword'},
   'content': {'type': 'text'},
   'page_number': {'type': 'integer'},
   'reading_order': {'type': 'integer'},
   'image_metadata': {'type': 'text', 'index': False},
   'table_metadata': {'type': 'text', 'index': False},
   'vetor': {'type': 'knn_vector',
    'dimension': 3072,
    'method': {'name': 'hnsw',
     'space_type': 'cosinesimil',
     'engine': 'lucene',
     'parameters': {'ef_construction': 100, 'm': 16}}}}},
 'settings': {'index.knn': True,
  'knn.algo_param.ef_search': 100,
  'number_of_shards': 2}}

In [45]:
CONTENT_INDEX

'content-pdfs'

In [44]:
os_client = OpenSearchClient()

2026-03-10 20:02:00 - src.services.opensearch - INFO - Connection with OpenSearch has been achieved. Version: 3.5.0


In [46]:
os_client.create_index(CONTENT_INDEX, content_mapping)

2026-03-10 20:02:42 - src.services.opensearch - INFO - Index 'content-pdfs' successfully created


True

In [47]:
for chunk in chunks_with_embeddings:
    if "embedding_vector" in chunk: 
        chunk["vetor"] = chunk.pop("embedding_vector")

In [48]:
chunks_with_embeddings[70]

{'book_id': 'atoms_confusion.pdf',
 'sec_0': 'Atoms of Confusion: The Eyes Do Not Lie',
 'sec_1': '4 RESULTS AND DISCUSSION',
 'sec_2': '4.3 RQ: To what extent do atoms of confusion\naffect the focus of attention?',
 'section_context': '4 RESULTS AND DISCUSSION > 4.3 RQ: To what extent do atoms of confusion\naffect the focus of attention?',
 'doc_type': 'text',
 'page_number': 7,
 'reading_order': 11,
 'content': "4 RESULTS AND DISCUSSION > 4.3 RQ: To what extent do atoms of confusion\naffect the focus of attention?\n\nIn the heatmaps, blue (col) areas are areas of the code that did not\nreceive much attention, while red (hot) areas represent the areas\nthat received most of the attention. By the developers' distribution\nof visual attention, and how the focus of attention changes over dis-\ntinct code elements, we can infer whether a particular code element\nis confusing. The analysis of heatmaps can assess the distribution\nof attention.The colors in the heatmap represent the relativ

In [49]:
inserted_chunks = os_client.bulk_insert(CONTENT_INDEX, chunks_with_embeddings)
print(f"Chunks inseridos: {inserted_chunks}")

2026-03-10 20:10:47 - src.services.opensearch - INFO - 217 documents inserted into index: 'content-pdfs'
Chunks inseridos: 217
